# CMIP6 Brazil Daily Data - Summary\n\nThis notebook reads all NetCDF files from `cmip6_brazil/daily/` and produces tables describing:\n1. **Variables available** per model and scenario\n2. **Grid, resolution, and time coverage** (detailed and grouped summary)

In [ ]:
VAR_NAMES = {
    
 'hur': 'Umidade Relativa',
 'hurs': 'Umidade Relativa Próxima à Superfície',
 'hus': 'Umidade Específica',
 'huss': 'Umidade Específica Próxima à Superfície',
 
 'pr': 'Precipitação',

 'psl': 'Pressão ao Nível do Mar',
 'sfcWind': 'Velocidade do Vento Próxima à Superfície',

 'ta': 'Temperatura do Ar',
 'tas': 'Temperatura do Ar Próxima à Superfície',
 'tasmax': 'Temperatura Máxima Diária do Ar Próxima à Superfície',
 'tasmin': 'Temperatura Mínima Diária do Ar Próxima à Superfície',
 'ts': 'Temperatura da Superfície'
}

In [3]:
import os
import re
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("cmip6_brazil/monthly")

# Map folder names to clean scenario labels
SCENARIO_MAP = {
    "historical": "Histórico",
    "ssp1_2_6": "SSP1-2.6",
    "ssp2_4_5": "SSP2-4.5",
    "ssp3_7_0": "SSP3-7.0",
}

# Parse filename pattern: {var}_day_{model}_{scenario}_{variant}_{grid}_{dates}.nc
FNAME_RE = re.compile(
    r"^(?P<variable>\w+)_Amon_(?P<model>[\w-]+)_(?P<scenario>\w+)_"
    r"(?P<variant>[\w]+)_(?P<grid>\w+)_(?P<date_start>\d{8})-(?P<date_end>\d{8})\.nc$"
)

def infer_grid_resolution(ds: xr.Dataset) -> str:
    try:
        lat_name, lon_name = 'lat', 'lon'
        lat_vals = np.asarray(ds[lat_name].values, dtype=float)
        lon_vals = np.asarray(ds[lon_name].values, dtype=float)
        if lat_vals.ndim == 1 and lon_vals.ndim == 1 and len(lat_vals) > 1 and len(lon_vals) > 1:
            lat_res = float(np.nanmedian(np.abs(np.diff(lat_vals))))
            lon_res = float(np.nanmedian(np.abs(np.diff(lon_vals))))
            return f"{lat_res:.2f}° x {lon_res:.2f}°"
    except Exception:
        pass
    return "N/A"

rows = []
for scenario_dir in sorted(DATA_DIR.iterdir()):
    if not scenario_dir.is_dir():
        continue
    scenario_label = SCENARIO_MAP.get(scenario_dir.name, scenario_dir.name)
    for nc_file in sorted(scenario_dir.glob("*.nc")):
        m = FNAME_RE.match(nc_file.name)
        if not m:
            continue
        info = m.groupdict()
        # Open file to get actual grid details
        with xr.open_dataset(nc_file) as ds:
            lat = ds["lat"].values
            lon = ds["lon"].values
            time = ds["time"].values
            lat_res = round(abs(float(lat[1] - lat[0])), 4) if len(lat) > 1 else None
            lon_res = round(abs(float(lon[1] - lon[0])), 4) if len(lon) > 1 else None
            nominal_resolution = ds.nominal_resolution

        # Convert cftime objects to datetime strings
        time_start = pd.Timestamp(time.min().isoformat()).date() if hasattr(time.min(), 'isoformat') else pd.Timestamp(time.min()).date()
        time_end = pd.Timestamp(time.max().isoformat()).date() if hasattr(time.max(), 'isoformat') else pd.Timestamp(time.max()).date()


        rows.append({
            "Cenário": scenario_label,
            "Modelo": info["model"],
            "Variável": info["variable"],
            "Nome da Variável": VAR_NAMES[info["variable"]],
            # "variant": info["variant"],
            "Unidade": ds[info["variable"]].attrs.get("units", "N/A"),
            
            "Dimensões": " x ".join(str(s) for s in ds[info["variable"]].shape),
            "Resolução Nominal": nominal_resolution,
            "Resolução Aproximada": infer_grid_resolution(ds),
            "Rótulo de Grid": info["grid"],
            
            # "n_lat": len(lat),
            # "n_lon": len(lon),
            # "lat_res": lat_res,
            # "lon_res": lon_res,
            # "lat_min": round(float(lat.min()), 2),
            # "lat_max": round(float(lat.max()), 2),
            # "lon_min": round(float(lon.min()), 2),
            # "lon_max": round(float(lon.max()), 2),
            "Início": str(time_start),
            "Fim": str(time_end),
            # "cell_methods": ds[info["variable"]].attrs.get("cell_methods", "N/A"),
            # "file": nc_file.name,
        })

    
df = pd.DataFrame(rows)
print(f"Parsed {len(df)} files across {df['Cenário'].nunique()} scenarios, "
      f"{df['Modelo'].nunique()} models, {df['Variável'].nunique()} variables")
df.head()

Parsed 568 files across 4 scenarios, 6 models, 13 variables


,Cenário,Modelo,Variável,Nome da Variável,Unidade,Dimensões,Resolução Nominal,Resolução Aproximada,Rótulo de Grid,Início,Fim
0,Histórico,ACCESS-ESM1-5,hur,Umidade Relativa,%,240 x 19 x 34 x 22,250 km,1.25° x 1.88°,gn,1995-01-16,2014-12-16
1,Histórico,GFDL-ESM4,hur,Umidade Relativa,%,240 x 19 x 43 x 32,100 km,1.00° x 1.25°,gr1,1995-01-16,2014-12-16
2,Histórico,IPSL-CM6A-LR,hur,Umidade Relativa,%,240 x 19 x 34 x 17,250 km,1.27° x 2.50°,gr,1995-01-16,2014-12-16
3,Histórico,IPSL-CM6A-LR,hurs,Umidade Relativa Próxima à Superfície,%,240 x 34 x 17,250 km,1.27° x 2.50°,gr,1995-01-16,2014-12-16
4,Histórico,ACCESS-ESM1-5,hus,Umidade Específica,1,240 x 19 x 34 x 22,250 km,1.25° x 1.88°,gn,1995-01-16,2014-12-16


## 1. Variables available per model and scenario\n\nCross-tabulation showing which variables are available for each model-scenario combination.

In [85]:
# Unique variables per model-scenario (collapse multiple time chunks into one entry)
var_by_model = (
    df.groupby(["Modelo", "Cenário"])["Variável"]
    .apply(lambda x: sorted(x.unique()))
    .reset_index()
    .rename(columns={"Variável": "variables"})
)

# Pivot: rows = model, columns = scenario, values = list of variables
pivot_vars = var_by_model.pivot(index="Modelo", columns="Cenário", values="variables")
# Order columns logically
col_order = [c for c in df["Cenário"].unique() if c in pivot_vars.columns]
pivot_vars = pivot_vars[col_order]

# Format lists as comma-separated strings for display
pivot_display = pivot_vars.map(
    lambda x: ", ".join(x) if isinstance(x, list) else "-"
)
pivot_display.style.set_caption("Variáveis Disponíveis por Modelo e Cenário")

Cenário,Histórico,SSP1-2.6,SSP2-4.5,SSP3-7.0
Modelo,,,,
ACCESS-ESM1-5,"huss, pr, psl, sfcWind, tas, tasmax, tasmin",-,-,-
GFDL-ESM4,"huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin","pr, psl, sfcWind, tas, tasmax","huss, pr, psl, sfcWind, tas, tasmax"
IPSL-CM6A-LR,"pr, sfcWind, tas, tasmax, tasmin","pr, psl, sfcWind, tas, tasmin","pr, psl, sfcWind, tas, tasmax, tasmin","pr, psl, sfcWind, tas, tasmax, tasmin"
MIROC6,"huss, pr, psl, sfcWind, tas, tasmax, tasmin","pr, psl, sfcWind, tas, tasmax, tasmin","pr, psl, sfcWind, tas, tasmax, tasmin","pr, psl, sfcWind, tas, tasmax, tasmin"
MRI-ESM2-0,"huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin"
NorESM2-MM,"huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin","huss, pr, psl, sfcWind, tas, tasmax, tasmin"


In [86]:
# Binary presence/absence matrix (variable x model, marked per scenario)
presence = df.drop_duplicates(subset=["Modelo", "Cenário", "Variável"])
cross = pd.crosstab(
    [presence["Modelo"], presence["Cenário"]],
    presence["Variável"],
)
cross = cross.clip(upper=1).replace({1: "X", 0: ""})
cross.style.set_caption("Presença de Variáveis por Modelo e Cenário (X = disponível)")

### 2b. Summary grouped by model\n\nGrid properties and overall time range per model (aggregated across all variables and scenarios).

In [5]:
df['Início'] = df['Início'].str.split("-",expand=True)[0]
df['Fim'] = df['Fim'].str.split("-",expand=True)[0]

In [6]:
# Summary by model: grid info is constant per model, time range spans all files
model_summary = (
    df.groupby("Modelo")
    .agg(
        
        scenarios=("Cenário", lambda x: ", ".join(sorted(x.unique()))),
        nominal_resolution=("Resolução Nominal", lambda x: ", ".join(sorted(x.unique()))),
        aprox_resolution = ("Resolução Aproximada", lambda x: ", ".join(sorted(x.unique()))),
        grid_label=("Rótulo de Grid", lambda x: ", ".join(sorted(x.unique()))),
        shape = ("Dimensões", lambda x: ", ".join(sorted(x.unique()))),
        
        
        # lat_res=("lat_res", "first"),
        # lon_res=("lon_res", "first"),
        # n_lat=("n_lat", "first"),
        # n_lon=("n_lon", "first"),
        # lat_range=("lat_min", lambda x: f"{x.min():.1f} to {df.loc[x.index, 'lat_max'].max():.1f}"),
        # lon_range=("lon_min", lambda x: f"{x.min():.1f} to {df.loc[x.index, 'lon_max'].max():.1f}"),
        time_start=("Início", "min"),
        time_end=("Fim", "max"),
        n_variables=("Variável", "nunique"),
        variaveis=("Variável", lambda x: ", ".join(sorted(x.unique()))),
        
        # n_files=("file", "count"),
    )
)
model_summary.columns = ['Cenários', 'Resolução Nominal', "Resolução Aproximada",  'Rótulo de Grid',  "Dimensão", 'Início', 'Fim',
       'Número de Variáveis', 'Variáveis']

model_summary.style.set_caption("Resumo por modelo")

,Cenários,Resolução Nominal,Resolução Aproximada,Rótulo de Grid,Dimensão,Início,Fim,Número de Variáveis,Variáveis
Modelo,,,,,,,,,
ACCESS-ESM1-5,Histórico,250 km,1.25° x 1.88°,gn,"240 x 19 x 34 x 22, 240 x 35 x 22",1995,2014,12,"hur, hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts"
GFDL-ESM4,"Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0",100 km,1.00° x 1.25°,gr1,"240 x 19 x 43 x 32, 240 x 43 x 32",1995,2060,12,"hur, hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts"
IPSL-CM6A-LR,"Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0",250 km,1.27° x 2.50°,gr,"240 x 19 x 34 x 17, 240 x 34 x 17",1995,2060,13,"hur, hurs, hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts"
MIROC6,"Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0",250 km,1.40° x 1.41°,gn,"240 x 19 x 31 x 29, 240 x 31 x 29",1995,2060,11,"hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts"
MRI-ESM2-0,"Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0",100 km,1.12° x 1.12°,gn,"240 x 19 x 38 x 36, 240 x 38 x 36",1995,2060,11,"hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts"
NorESM2-MM,"Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0",100 km,0.94° x 1.25°,gn,"240 x 19 x 45 x 33, 240 x 45 x 33",1995,2060,9,"hus, huss, pr, ps, psl, sfcWind, ta, tas, ts"


In [7]:
latex_table = (
    model_summary.style
    .set_caption("Resumo por modelo")
    .to_latex(hrules=True)
)

print(latex_table)

\begin{table}
\caption{Resumo por modelo}
\begin{tabular}{llllllllrl}
\toprule
 & Cenários & Resolução Nominal & Resolução Aproximada & Rótulo de Grid & Dimensão & Início & Fim & Número de Variáveis & Variáveis \\
Modelo &  &  &  &  &  &  &  &  &  \\
\midrule
ACCESS-ESM1-5 & Histórico & 250 km & 1.25° x 1.88° & gn & 240 x 19 x 34 x 22, 240 x 35 x 22 & 1995 & 2014 & 12 & hur, hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts \\
GFDL-ESM4 & Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0 & 100 km & 1.00° x 1.25° & gr1 & 240 x 19 x 43 x 32, 240 x 43 x 32 & 1995 & 2060 & 12 & hur, hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts \\
IPSL-CM6A-LR & Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0 & 250 km & 1.27° x 2.50° & gr & 240 x 19 x 34 x 17, 240 x 34 x 17 & 1995 & 2060 & 13 & hur, hurs, hus, huss, pr, ps, psl, sfcWind, ta, tas, tasmax, tasmin, ts \\
MIROC6 & Histórico, SSP1-2.6, SSP2-4.5, SSP3-7.0 & 250 km & 1.40° x 1.41° & gn & 240 x 19 x 31 x 29, 240 x 31 x 29 & 1995 & 2060 & 1

In [163]:
import dataframe_image as dfi

styled = (
    model_summary.style
    .set_caption("Resumo por modelo")

)

dfi.export(
   styled,
    "model_summary.png",
    table_conversion="matplotlib"
)